In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE = Path('/content/drive/MyDrive/bigcontest-2026')
OUT = BASE / 'data' / 'processed'

# 기존 분석에서 만든 파일 불러오기
features = pd.read_csv(OUT / 'grid_features.csv')
types = pd.read_csv(OUT / 'grid_types.csv')

print("격자 지표:", features.shape)
print("상권 유형:", types.shape)

print("grid_features 컬럼:")
print(features.columns.tolist())

print("grid_types 컬럼:")
print(types.columns.tolist())

In [ ]:
print("grid_features.csv 컬럼 목록")
print(features.columns.tolist())

print("\ngrid_types.csv 컬럼 목록")
print(types.columns.tolist())

In [ ]:
# 지역과 격자 좌표를 기준으로 연결
KEY = ['region', 'X_COORD', 'Y_COORD']

# 폭염·폭우 분석에 필요한 지표
required_cols = [
    'elder_share',
    'work_share',
    'pop',
    'n_stores',
    'tour_share',
    'n_tour',
    'weekend_ratio'
]

# 두 파일에 공통으로 있는 분석 지표 확인
print("grid_features 컬럼:", features.columns.tolist())
print("grid_types 컬럼:", types.columns.tolist())

# features에서 가져올 열만 선택
use_cols = KEY + required_cols

# 주요 시설 정보가 있으면 추가
for col in ['has_golf', 'has_lego']:
    if col in features.columns:
        use_cols.append(col)

# types에 이미 들어 있는 지표는 제외하고 연결
features_to_merge = features[use_cols].copy()

cols_to_drop = [
    col for col in required_cols + ['has_golf', 'has_lego']
    if col in types.columns
]

types_clean = types.drop(columns=cols_to_drop)

df = types_clean.merge(
    features_to_merge,
    on=KEY,
    how='left',
    validate='one_to_one'
)

# 숫자형으로 변환
for col in required_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

for col in ['has_golf', 'has_lego']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

print("\n연결된 격자 수:", len(df))
print("\n지역별 격자 수:")
print(df['region'].value_counts())

print("\n최종 컬럼 목록:")
print(df.columns.tolist())

display(df.head())

In [ ]:
def rank01(series):
    """
    지표를 지역 내 백분위 순위로 변환합니다.
    결과 범위: 0~1
    """
    x = pd.to_numeric(series, errors='coerce')

    if x.notna().any():
        x = x.fillna(x.median())
    else:
        x = x.fillna(0)

    return x.rank(pct=True)


df['heat_index'] = np.nan
df['rain_index'] = np.nan

for region, idx in df.groupby('region').groups.items():

    g = df.loc[idx]

    # -------------------------
    # 폭염 취약 가능성 지수
    # -------------------------
    heat_score = (
        0.30 * rank01(g['elder_share']) +
        0.25 * rank01(g['work_share']) +
        0.25 * rank01(g['pop']) +
        0.20 * rank01(g['n_stores'])
    )

    df.loc[idx, 'heat_index'] = heat_score.to_numpy()

    # -------------------------
    # 폭우 취약 가능성 지수
    # -------------------------
    special = pd.Series(0.0, index=g.index)

    if 'has_golf' in g.columns:
        special += g['has_golf'].fillna(0)

    if 'has_lego' in g.columns:
        special += g['has_lego'].fillna(0)

    rain_score = (
        0.35 * rank01(g['tour_share']) +
        0.25 * rank01(g['n_tour']) +
        0.20 * rank01(g['weekend_ratio']) +
        0.10 * rank01(g['n_stores']) +
        0.10 * rank01(special)
    )

    df.loc[idx, 'rain_index'] = rain_score.to_numpy()


print("폭염 지수 요약")
display(df['heat_index'].describe())

print("폭우 지수 요약")
display(df['rain_index'].describe())

In [ ]:
!pip -q install pyproj

In [ ]:
# 한글 폰트 설치
!apt-get -qq -y install fonts-nanum

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 나눔고딕 폰트 설정
font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'

fm.fontManager.addfont(font_path)
font_name = fm.FontProperties(fname=font_path).get_name()

plt.rcParams['font.family'] = font_name
plt.rcParams['axes.unicode_minus'] = False

print("설정된 한글 폰트:", font_name)

In [ ]:
!pip -q install contextily

In [ ]:
# ============================================================
# 취약성 위험 등급 분류 함수
# ============================================================

def make_risk_level(df, index_col):
    df = df.copy()

    # 취약성 지수를 기준으로 5개 등급 분류
    df["risk_level"] = pd.qcut(
        df[index_col].rank(method="first"),
        q=5,
        labels=[
            "매우 낮음",
            "낮음",
            "보통",
            "높음",
            "매우 높음"
        ]
    )

    return df

In [ ]:
# 폭우 취약성 지수 분포 확인

print("===== 폭우 취약성 지수 분포 =====")

print(
    df.groupby("region")["rain_index"]
    .describe()
    .round(4)
)

print("\n===== 폭우 취약성 지수 고유값 개수 =====")

print(
    df.groupby("region")["rain_index"]
    .nunique()
)

print("\n===== 폭우 취약성 등급별 격자 수 =====")

for region in ["강남", "춘천"]:

    g = df[
        df["region"].astype(str) == region
    ].copy()

    g = make_risk_level(g, "heat_index")

    print(f"\n[{region}]")

    print(
        g["risk_level"]
        .value_counts(sort=False)
    )

In [ ]:
# ============================================================
# 폭우 취약성 등급 분류 수정
# 0인 격자와 양수인 격자를 구분
# ============================================================

def make_rain_risk_level(data, score_col="rain_index"):

    result = data.copy()

    result["risk_level"] = "낮음"

    # 지수가 0보다 큰 격자만 별도로 분류
    positive_mask = result[score_col] > 0
    positive = result.loc[positive_mask, score_col]

    if len(positive) > 0:

        percentile = positive.rank(
            pct=True,
            method="average"
        )

        result.loc[positive_mask, "risk_level"] = pd.cut(
            percentile,
            bins=[0, 0.50, 0.75, 0.90, 1.00],
            labels=RISK_LEVELS,
            include_lowest=True
        ).astype(str)

    return result

In [ ]:
# 폭우 취약성 등급별 격자 수 확인

for region in ["강남", "춘천"]:

    g = df[
        df["region"].astype(str) == region
    ].copy()

    g = make_risk_level(g, "rain_index")

    print(f"\n[{region}]")
    print(g["risk_level"].value_counts(sort=False))

In [ ]:
# ============================================================
# 폭염 분석 결과 확인 (age_effects 없이도 실행 가능)
# ============================================================

print("===== 지도 데이터의 지역별 상권 유형 =====")
print(
    df.groupby(["region", "type"])
    .size()
    .to_string()
)

# age_effects가 있을 때만 폭염 효과 확인
if 'age_effects' in globals():

    print("\n===== 춘천 60대 이상 결과 =====")
    print(
        age_effects[
            (age_effects["region"].astype(str).str.contains("춘천"))
            & (age_effects["age_grp"].astype(str).str.contains("60"))
        ][["region", "type", "age_grp", "heat", "heat_p"]]
        .to_string(index=False)
    )

    print("\n===== 폭염 효과의 유의성 (상위 20) =====")
    print(
        age_effects[["region", "type", "age_grp", "heat", "heat_p"]]
        .sort_values("heat_p")
        .head(20)
        .to_string(index=False)
    )
else:
    print("\n(age_effects 없음 → 06 분석 결과 출력은 건너뜁니다)")

In [ ]:
# ============================================================
# 15. 폭염·폭우 취약 상권 지도 (전체 수정 코드)
#
# 지도 표현 방식
#   1. 강남구 전체: 회색 원형
#   2. 춘천시 전체: 점수에 따른 원형 버블
#   3. 폭염 시내 확대: 격자형
#   4. 폭우 시내 확대: 격자형
#
# 전체 지도:
#   - 점수가 높을수록 원이 커짐
#   - 점수가 높을수록 색이 진해짐
#   - 높은 점수의 원이 위에 표시됨
#
# 확대 지도:
#   - 원형 버블 대신 격자 사용
#   - 폭염: 빨간색 계열
#   - 폭우: 파란색 계열
#   - 점수가 높을수록 색이 진해짐
#
# 결과물:
#   outputs/maps/heat_vulnerability_map.png
#   outputs/maps/rain_vulnerability_map.png
#   outputs/maps/grid_vulnerability_scores.csv
# ============================================================

import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize, LinearSegmentedColormap
from matplotlib.patches import Polygon as MPoly, Rectangle
from matplotlib.collections import PatchCollection

from pyproj import Transformer

warnings.filterwarnings("ignore")


# ============================================================
# 1. 한글 폰트
# ============================================================

nanum = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"

if Path(nanum).exists():
    fm.fontManager.addfont(nanum)
    plt.rcParams["font.family"] = [
        fm.FontProperties(fname=nanum).get_name(),
        "DejaVu Sans"
    ]

plt.rcParams["axes.unicode_minus"] = False


# ============================================================
# 2. 경로
# ============================================================

BASE = Path("/content/drive/MyDrive/bigcontest-2026")

DATA_DIR = BASE / "data" / "processed"
OUTPUT_DIR = BASE / "outputs" / "maps"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GRID_TYPES_PATH = DATA_DIR / "grid_types.csv"
STORES_PATH = DATA_DIR / "stores_grid.csv"

GEOJSON_PATH = (
    BASE / "data" / "external" / "adm"
    / "adm_gangnam_chuncheon_2025.geojson"
)

for p in [
    GRID_TYPES_PATH,
    STORES_PATH,
    GEOJSON_PATH
]:
    if not p.exists():
        raise FileNotFoundError(
            f"파일을 찾을 수 없습니다: {p}"
        )


# ============================================================
# 3. 데이터 불러오기
# ============================================================

grid = pd.read_csv(GRID_TYPES_PATH)

stores = pd.read_csv(
    STORES_PATH,
    low_memory=False
)

print("grid_types:", grid.shape)
print("stores_grid:", stores.shape)

for col in [
    "X_COORD",
    "Y_COORD",
    "elder_share",
    "n_stores",
    "has_golf",
    "has_lego"
]:
    grid[col] = pd.to_numeric(
        grid[col],
        errors="coerce"
    )

for col in [
    "grid_x",
    "grid_y"
]:
    stores[col] = pd.to_numeric(
        stores[col],
        errors="coerce"
    )


# 좌표 연결용 키
grid["X_JOIN"] = grid["X_COORD"].round(2)
grid["Y_JOIN"] = grid["Y_COORD"].round(2)

stores["X_JOIN"] = stores["grid_x"].round(2)
stores["Y_JOIN"] = stores["grid_y"].round(2)


# 격자에 연결된 점포만 사용
in_grid = (
    stores["in_grid"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(["true", "1"])
)

stores = stores[
    in_grid
].dropna(
    subset=["X_JOIN", "Y_JOIN"]
).copy()

stores["ind_grp"] = (
    stores["ind_grp"]
    .astype(str)
    .str.strip()
)

print("\n[지역별 격자 수]")
print(grid["region"].value_counts())

chun = grid["region"].eq("춘천")
gang = grid["region"].eq("강남")


# ============================================================
# 4. 폭염 점수 계산
# ============================================================

HEAT_EFFECT = 0.189283

grid["heat_score"] = 0.0

grid.loc[chun, "heat_score"] = (
    grid.loc[chun, "elder_share"]
    .fillna(0)
    .clip(0, 1)
    * HEAT_EFFECT
)


# ============================================================
# 5. 폭우 점수 계산
# ============================================================

RAIN_EFFECT = {
    "야외레저": 0.534366,
    "자동차": 0.079918,
    "카페·제과": 0.084128
}

stores["rain_loss"] = (
    stores["ind_grp"]
    .map(RAIN_EFFECT)
    .fillna(0)
)

# 강남은 유의한 감소 효과가 없으므로 0
stores.loc[
    stores["region"] != "춘천",
    "rain_loss"
] = 0


# 격자별 업종 피해율 합계
loss_sum = (
    stores.groupby(
        ["region", "X_JOIN", "Y_JOIN"]
    )["rain_loss"]
    .sum()
    .rename("rain_loss_sum")
    .reset_index()
)

grid = grid.drop(
    columns=["rain_loss_sum"],
    errors="ignore"
).merge(
    loss_sum,
    on=["region", "X_JOIN", "Y_JOIN"],
    how="left"
)

grid["rain_loss_sum"] = (
    grid["rain_loss_sum"]
    .fillna(0)
)


# 격자 내 점포 수로 나눈 평균 피해율
grid["rain_score"] = (
    grid["rain_loss_sum"]
    / grid["n_stores"].clip(lower=1)
).clip(
    0,
    RAIN_EFFECT["야외레저"]
)


# 골프장·레고랜드 격자는 야외레저 피해율 적용
special = (
    (grid["has_golf"] > 0)
    | (grid["has_lego"] > 0)
)

grid.loc[
    special & chun,
    "rain_score"
] = RAIN_EFFECT["야외레저"]


# 강남 점수 0
grid.loc[
    gang,
    ["heat_score", "rain_score"]
] = 0.0


print("\n[폭염 점수 %]")
print(
    grid.groupby("region")["heat_score"]
    .agg(["mean", "max"])
    .mul(100)
    .round(3)
)

print("\n[폭우 점수 %]")
print(
    grid.groupby("region")["rain_score"]
    .agg(["mean", "max"])
    .mul(100)
    .round(3)
)


# ============================================================
# 6. 행정동 경계
# ============================================================

INK = "#1d2430"
SUB = "#5b6472"
LAND = "#eef0f3"

tf = Transformer.from_crs(
    "EPSG:4326",
    "EPSG:5179",
    always_xy=True
)

with open(
    GEOJSON_PATH,
    encoding="utf-8"
) as f:
    gj = json.load(f)


def dongs(sgg):

    out = []

    for feature in gj["features"]:

        if sgg not in feature["properties"]["adm_nm"]:
            continue

        geometry = feature["geometry"]

        polygons = (
            geometry["coordinates"]
            if geometry["type"] == "MultiPolygon"
            else [geometry["coordinates"]]
        )

        rings = []

        for polygon in polygons:

            ring = np.array(polygon[0])

            x, y = tf.transform(
                ring[:, 0],
                ring[:, 1]
            )

            rings.append(
                np.c_[x, y]
            )

        out.append(
            (
                feature["properties"]["adm_nm"].split()[-1],
                rings
            )
        )

    return out


D = {
    "강남": dongs("강남구"),
    "춘천": dongs("춘천시")
}


# ============================================================
# 7. 지도 기본 함수
# ============================================================

def base(ax, reg, bounds=None):

    patches = [
        MPoly(
            ring,
            closed=True
        )
        for _, rings in D[reg]
        for ring in rings
    ]

    ax.add_collection(
        PatchCollection(
            patches,
            facecolor=LAND,
            edgecolor="#9ba1aa",
            linewidth=0.55,
            zorder=1
        )
    )

    all_coords = np.vstack([
        ring
        for _, rings in D[reg]
        for ring in rings
    ])

    if bounds is None:

        x0 = all_coords[:, 0].min()
        x1 = all_coords[:, 0].max()

        y0 = all_coords[:, 1].min()
        y1 = all_coords[:, 1].max()

    else:

        (x0, x1), (y0, y1) = bounds

    ax.set_xlim(x0, x1)
    ax.set_ylim(y0, y1)
    ax.set_aspect("equal")
    ax.set_axis_off()


def call(
    ax,
    x,
    y,
    text,
    color,
    dx,
    dy,
    fs=12
):

    ax.annotate(
        text,
        (x, y),
        xytext=(dx, dy),
        textcoords="offset points",
        fontsize=fs,
        fontweight="bold",
        color=color,
        zorder=100,
        bbox=dict(
            boxstyle="round,pad=0.4",
            fc="white",
            ec=color,
            lw=1.3
        ),
        arrowprops=dict(
            arrowstyle="-",
            color=color,
            lw=1.3
        )
    )


def scalebar(ax, km):

    x0, x1 = ax.get_xlim()
    y0, y1 = ax.get_ylim()

    length = km * 1000

    bx = x0 + (x1 - x0) * 0.05
    by = y0 + (y1 - y0) * 0.04

    ax.plot(
        [bx, bx + length],
        [by, by],
        color=INK,
        lw=3,
        solid_capstyle="butt",
        zorder=50
    )

    ax.text(
        bx + length / 2,
        by + (y1 - y0) * 0.015,
        f"{km}km",
        ha="center",
        fontsize=10,
        color=INK,
        zorder=50
    )


# ============================================================
# 8. 버블 크기 설정
# ============================================================

MIN_BUBBLE_SIZE = 2
MAX_BUBBLE_SIZE = 3000

BUBBLE_ALPHA = 0.72

SIZE_POWER = 1.5


def bubble_sizes(scores, max_score,
                 min_size=MIN_BUBBLE_SIZE,
                 max_size=MAX_BUBBLE_SIZE):
    """draw_bubble_map과 동일한 공식으로 원 크기를 계산 (범례용)"""
    scores = np.asarray(scores, dtype=float)
    if max_score > 0:
        return (
            min_size
            + (max_size - min_size)
            * np.power(scores / max_score, SIZE_POWER)
        )
    return np.full(scores.shape, float(min_size))


# ============================================================
# 9. 전체 지도용 원형 버블 함수
# ============================================================

def draw_bubble_map(
    ax,
    data,
    score_col,
    cmap,
    norm,
    min_size=MIN_BUBBLE_SIZE,
    max_size=MAX_BUBBLE_SIZE,
    alpha=BUBBLE_ALPHA
):

    data = data.dropna(
        subset=[
            "X_COORD",
            "Y_COORD",
            score_col
        ]
    ).copy()

    if data.empty:
        return 0.0

    scores = (
        data[score_col]
        .fillna(0)
        .clip(lower=0)
        .to_numpy(dtype=float)
    )

    max_score = scores.max()

    if max_score > 0:

        relative = scores / max_score

        sizes = (
            min_size
            + (max_size - min_size)
            * np.power(relative, SIZE_POWER)
        )

    else:

        sizes = np.full(
            len(scores),
            min_size,
            dtype=float
        )

    colors = cmap(norm(scores))

    # 낮은 점수부터 높은 점수 순으로 표시
    order = np.argsort(
        scores,
        kind="stable"
    )

    x = data["X_COORD"].to_numpy()[order]
    y = data["Y_COORD"].to_numpy()[order]

    sizes = sizes[order]
    colors = colors[order]

    n = len(order)

    for i in range(n):

        ax.scatter(
            [x[i]],
            [y[i]],
            s=float(sizes[i]),
            c=[colors[i]],
            marker="o",
            alpha=alpha,
            edgecolors="none",
            linewidths=0,
            zorder=3 + i / max(n, 1)
        )

    return float(max_score)


# ============================================================
# 10. 확대 지도용 격자 함수
# ============================================================

def draw_grid_map(
    ax,
    data,
    score_col,
    cmap,
    norm,
    grid_size=180,
    alpha=0.90
):
    """
    확대 지도에서는 원형 버블 대신 사각형 격자를 사용합니다.

    grid_size:
        격자의 한 변 길이(미터 단위)

    색상:
        점수가 낮으면 연한 색
        점수가 높으면 진한 색

    겹침:
        낮은 점수부터 높은 점수 순서로 표시
    """

    data = data.dropna(
        subset=[
            "X_COORD",
            "Y_COORD",
            score_col
        ]
    ).copy()

    if data.empty:
        return

    data[score_col] = (
        data[score_col]
        .fillna(0)
        .clip(lower=0)
    )

    # 높은 점수의 격자를 나중에 그리기
    data = data.sort_values(
        score_col,
        ascending=True,
        kind="stable"
    )

    patches = []
    values = []

    half = grid_size / 2

    for _, row in data.iterrows():

        x = row["X_COORD"]
        y = row["Y_COORD"]
        score = row[score_col]

        square = Rectangle(
            (x - half, y - half),
            grid_size,
            grid_size
        )

        patches.append(square)
        values.append(score)

    # PatchCollection의 배열에 점수를 넣어
    # 격자별로 점수에 맞는 색상 지정
    collection = PatchCollection(
        patches,
        cmap=cmap,
        norm=norm,
        edgecolor="none",
        linewidth=0,
        alpha=alpha,
        zorder=5
    )

    collection.set_array(
        np.asarray(values)
    )

    ax.add_collection(collection)


# ============================================================
# 11. 지도 범위 설정
# ============================================================

ch = grid[chun].copy()
gn = grid[gang].copy()

city = ch[
    ch["type"] == "시내 생활형"
]

lego = ch[
    ch["has_lego"] > 0
].iloc[0]

golf = ch[
    ch["has_golf"] > 0
]


# 시내 확대 범위
zb = (
    (
        min(
            city["X_COORD"].quantile(0.03),
            lego["X_COORD"]
        ) - 700,

        city["X_COORD"].quantile(0.97) + 500
    ),

    (
        city["Y_COORD"].quantile(0.03) - 500,

        max(
            city["Y_COORD"].quantile(0.97),
            lego["Y_COORD"]
        ) + 700
    )
)


# 중앙시장 위치
mx, my = tf.transform(
    127.7279,
    37.8803
)

mkt = ch[
    (ch["type"] == "낮·고령형")
    & (
        np.hypot(
            ch["X_COORD"] - mx,
            ch["Y_COORD"] - my
        ) < 800
    )
]


# 색상 기준
norm = Normalize(
    vmin=0,
    vmax=0.10,
    clip=False
)


# ============================================================
# 12. 폭염·폭우 지도 생성
# ============================================================

def make(kind):

    if kind == "rain":

        col = "rain_score"
        acc = "#1f5fa8"

        cmap = LinearSegmentedColormap.from_list(
            "rain_cmap",
            [
                "#d6e6f5",
                "#8fc5e8",
                "#4292c6",
                "#2171b5",
                "#08306b"
            ]
        )

        cmap.set_over("#041f45")

        head = (
            "호우 피해는 춘천 남쪽 골프장 지대와 "
            "레고랜드에 집중"
        )

        sub = (
            "06 분석: 춘천 호우일 야외레저 -53.4% · "
            "자동차 -8.0% · 카페·제과 -8.4% "
            "(FDR 보정 후 유의). "
            "강남은 같은 기준에서 유의한 감소 없음."
        )

    else:

        col = "heat_score"
        acc = "#a3260f"

        cmap = LinearSegmentedColormap.from_list(
            "heat_cmap",
            [
                "#fde9c9",
                "#fdae61",
                "#f46d43",
                "#d73027",
                "#a3260f"
            ]
        )

        cmap.set_over("#67000d")

        head = (
            "폭염 취약 상권은 고령층 비중이 높은 "
            "춘천 외곽과 중앙시장 일대"
        )

        sub = (
            "06 분석: 폭염일 춘천 60대 이상 "
            "여가·외출 소비 -18.9% "
            "(다른 연령·강남은 유의하지 않음). "
            "격자 점수 = 60대 이상 비중 × 18.9%."
        )

    # --------------------------------------------------------
    # 캔버스
    # --------------------------------------------------------

    FW, FH = 16, 19

    fig = plt.figure(figsize=(FW, FH))
    fig.patch.set_facecolor("white")

    axg = fig.add_axes([0.03, 0.50, 0.45, 0.38])
    axc = fig.add_axes([0.52, 0.50, 0.45, 0.38])
    axz = fig.add_axes([0.03, 0.08, 0.94, 0.36])

    # ========================================================
    # 12-1. 강남구 전체
    # ========================================================

    base(axg, "강남")

    axg.scatter(
        gn["X_COORD"],
        gn["Y_COORD"],
        c="#c4c8ce",
        s=8,
        marker="o",
        linewidths=0,
        alpha=0.75,
        zorder=3
    )

    axg.set_title(
        "강남구",
        fontsize=15,
        fontweight="bold",
        color=INK,
        loc="left"
    )

    axg.text(
        0.97,
        0.95,
        "통계적으로 유의한\n감소 없음",
        transform=axg.transAxes,
        fontsize=14,
        fontweight="bold",
        color=SUB,
        ha="right",
        va="top",
        bbox=dict(
            boxstyle="round,pad=0.5",
            fc="white",
            ec="#c4c8ce"
        ),
        zorder=100
    )

    scalebar(axg, 2)

    # ========================================================
    # 12-2. 춘천시 전체: 원형 버블 유지
    # ========================================================

    base(axc, "춘천")

    max_score = draw_bubble_map(
        ax=axc,
        data=ch,
        score_col=col,
        cmap=cmap,
        norm=norm,
        min_size=MIN_BUBBLE_SIZE,
        max_size=MAX_BUBBLE_SIZE,
        alpha=BUBBLE_ALPHA
    )

    if kind == "rain":

        axc.scatter(
            golf["X_COORD"],
            golf["Y_COORD"],
            s=170,
            marker="^",
            color=acc,
            edgecolor="white",
            linewidth=1.3,
            zorder=100
        )

        gi = golf.loc[
            golf["Y_COORD"].idxmin()
        ]

        call(
            axc,
            gi["X_COORD"],
            gi["Y_COORD"],
            "골프장 12곳\n호우 시 매출 -53.4%",
            acc,
            -200,
            -5
        )

    else:

        top = ch[
            ch["heat_score"] >= 0.07
        ]

        if not top.empty:

            axc.scatter(
                top["X_COORD"],
                top["Y_COORD"],
                s=45,
                facecolors="none",
                edgecolors=acc,
                linewidths=1.2,
                zorder=100
            )

            call(
                axc,
                top["X_COORD"].iloc[0],
                top["Y_COORD"].iloc[0],
                (
                    "60대 비중 37% 이상 외곽 상권\n"
                    f"예상 -7% 이상 ({len(top)}칸)"
                ),
                acc,
                -215,
                45
            )

    # 확대 범위 사각형
    axc.add_patch(
        Rectangle(
            (zb[0][0], zb[1][0]),
            zb[0][1] - zb[0][0],
            zb[1][1] - zb[1][0],
            fill=False,
            edgecolor=INK,
            linewidth=1.4,
            linestyle="--",
            zorder=101
        )
    )

    axc.text(
        zb[0][1],
        zb[1][0],
        "  ↓ 아래 확대",
        fontsize=11,
        color=INK,
        va="top",
        zorder=102
    )

    axc.set_title(
        "춘천시",
        fontsize=15,
        fontweight="bold",
        color=INK,
        loc="left"
    )

    scalebar(axc, 10)

    fig.text(
        0.03,
        0.475,
        (
            "※ 위 두 지도는 축척이 다릅니다 "
            "(축척 막대 참고). "
            "실제 면적: 강남구 약 40㎢ · "
            "춘천시 약 1,116㎢ (약 28배)"
        ),
        fontsize=10.5,
        color=SUB
    )

    # ========================================================
    # 12-3. 춘천 시내 확대: 격자형으로 복구
    # ========================================================

    zx = zb[0][1] - zb[0][0]
    zy = zb[1][1] - zb[1][0]

    ar = (
        (0.94 * FW)
        / (0.36 * FH)
    )

    cx0 = (
        zb[0][0] + zb[0][1]
    ) / 2

    half = max(
        zx,
        zy * ar
    ) / 2

    zb2 = (
        (cx0 - half, cx0 + half),
        (zb[1][0], zb[1][1])
    )

    base(
        axz,
        "춘천",
        zb2
    )

    zc = ch[
        ch["X_COORD"].between(*zb2[0])
        & ch["Y_COORD"].between(*zb2[1])
    ].copy()

    # --------------------------------------------------------
    # 핵심: 확대 지도는 원형이 아닌 사각형 격자
    # --------------------------------------------------------

    draw_grid_map(
        ax=axz,
        data=zc,
        score_col=col,
        cmap=cmap,
        norm=norm,
        grid_size=180,
        alpha=0.90
    )

    # 행정동 이름
    for name, rings in D["춘천"]:

        for ring in rings:

            cx = ring[:, 0].mean()
            cy = ring[:, 1].mean()

            if (
                zb2[0][0] < cx < zb2[0][1]
                and zb2[1][0] < cy < zb2[1][1]
            ):

                axz.text(
                    cx,
                    cy,
                    name,
                    fontsize=9,
                    color="#7b828c",
                    ha="center",
                    zorder=2
                )

                break

    # ========================================================
    # 12-4. 확대 지도 주석
    # ========================================================

    if kind == "rain":

        axz.scatter(
            [lego["X_COORD"]],
            [lego["Y_COORD"]],
            s=420,
            marker="*",
            color=acc,
            edgecolor="white",
            linewidth=1.5,
            zorder=110
        )

        call(
            axz,
            lego["X_COORD"],
            lego["Y_COORD"],
            "레고랜드  -53.4%",
            acc,
            -150,
            30
        )

        hot = zc[
            (zc["rain_score"] >= 0.05)
            & ~(
                (zc["has_golf"] > 0)
                | (zc["has_lego"] > 0)
            )
        ]

        if not hot.empty:

            call(
                axz,
                hot["X_COORD"].median(),
                hot["Y_COORD"].median(),
                (
                    "카페·자동차 밀집 칸\n"
                    f"예상 -5~10%대 ({len(hot)}칸)"
                ),
                acc,
                160,
                -60,
                11.5
            )

    else:

        if not mkt.empty:

            call(
                axz,
                mkt["X_COORD"].mean(),
                mkt["Y_COORD"].mean(),
                (
                    "중앙시장 일대 (낮·고령형)\n"
                    f"60대 비중 "
                    f"{mkt['elder_share'].mean():.0%}"
                    f" → 예상 "
                    f"-{mkt['heat_score'].mean():.1%}"
                ),
                acc,
                -260,
                -60
            )

    axz.set_title(
        "춘천 시내 확대",
        fontsize=15,
        fontweight="bold",
        color=INK,
        loc="left"
    )

    # ========================================================
    # 12-5. 범례
    # ========================================================

    cax = fig.add_axes(
        [0.62, 0.035, 0.30, 0.012]
    )

    cb = fig.colorbar(
        ScalarMappable(
            norm=norm,
            cmap=cmap
        ),
        cax=cax,
        orientation="horizontal",
        extend="max"
    )

    cb.set_ticks([0, 0.02, 0.05, 0.10])

    cb.set_ticklabels([
        "0%",
        "2%",
        "5%",
        "10%+"
    ])

    cb.set_label(
        "예상 매출 감소율 (색상)",
        fontsize=11
    )

    fig.text(
        0.03,
        0.035,
        (
            "※ 전체 지도는 원 크기와 색상으로, "
            "확대 지도는 격자 색상으로 예상 감소율을 표현합니다."
        ),
        fontsize=9.5,
        color=SUB
    )

    fig.text(
        0.03,
        0.955,
        head,
        fontsize=22,
        fontweight="bold",
        color=INK
    )

    fig.text(
        0.03,
        0.93,
        sub,
        fontsize=11.5,
        color=SUB
    )

    # ========================================================
    # 12-5b. 기호 설명 (그래프의 원·도형이 무엇인지 설명)
    # ========================================================

    axl = fig.add_axes([0.03, -0.23, 0.94, 0.19])
    axl.set_xlim(0, 1)
    axl.set_ylim(0, 1)
    axl.set_axis_off()

    axl.add_patch(
        Rectangle(
            (0, 0), 1, 1,
            transform=axl.transAxes,
            fill=False,
            edgecolor="#c4c8ce",
            linewidth=1.2
        )
    )

    # ---- 왼쪽: 춘천 전체 지도 원 크기·색 기준 ----
    axl.text(
        0.015, 0.90,
        "춘천시 전체 지도의 원 (원 1개 = 격자 1칸)",
        fontsize=12.5, fontweight="bold",
        color=INK, va="center"
    )
    axl.text(
        0.015, 0.78,
        "원이 클수록 · 색이 진할수록 예상 매출 감소율이 큼",
        fontsize=11, color=SUB, va="center"
    )

    if max_score > 0:

        picks = [
            v for v in (0.01, 0.02, 0.05, 0.10, 0.20, 0.30)
            if v < max_score * 0.7
        ]

        ref_vals = (
            [picks[0], picks[-1]]
            if len(picks) >= 2 else picks
        ) + [max_score]

        ref_sizes = bubble_sizes(ref_vals, max_score)
        ref_x = np.linspace(0.07, 0.30, len(ref_vals))

        for xv, v, sv in zip(ref_x, ref_vals, ref_sizes):

            axl.scatter(
                [xv], [0.43],
                s=float(sv),
                color=cmap(norm(v)),
                alpha=BUBBLE_ALPHA,
                edgecolors="none",
                zorder=3
            )

            label = (
                f"{v:.0%}" if v < max_score
                else f"최대 {v:.1%}"
            )

            axl.text(
                xv, 0.08, label,
                ha="center", va="center",
                fontsize=11, color=INK
            )

    # ---- 오른쪽: 기호별 의미 ----
    items = [
        ("gray",
         "강남구 격자 1칸 (유의한 감소 없음 → 모두 회색, 점수 0)")
    ]

    if kind == "heat" and not top.empty:
        items.append((
            "ring",
            "빨간 테두리 원: 60대 비중 37% 이상 외곽 상권 "
            f"(예상 -7% 이상, {len(top)}칸)"
        ))

    if kind == "rain":
        items.append((
            "tri",
            "골프장 격자 (호우 시 예상 -53.4%)"
        ))
        items.append((
            "star",
            "레고랜드 격자 (호우 시 예상 -53.4%)"
        ))

    items.append((
        "zoom",
        "점선 사각형: 아래 '춘천 시내 확대' 지도의 범위"
    ))
    items.append((
        "cell",
        "확대 지도의 사각형 = 격자 1칸 (색이 진할수록 감소율 큼)"
    ))

    lx = 0.47
    ly = 0.88

    for key, text in items:

        if key == "gray":
            axl.scatter(
                [lx], [ly], s=40, c="#c4c8ce", alpha=0.9,
                edgecolors="none", zorder=3
            )

        elif key == "ring":
            axl.scatter(
                [lx], [ly], s=110, facecolors="none",
                edgecolors=acc, linewidths=1.6, zorder=3
            )

        elif key == "tri":
            axl.scatter(
                [lx], [ly], s=170, marker="^", color=acc,
                edgecolor="white", linewidth=1.3, zorder=3
            )

        elif key == "star":
            axl.scatter(
                [lx], [ly], s=420, marker="*", color=acc,
                edgecolor="white", linewidth=1.5, zorder=3
            )

        elif key == "zoom":
            axl.add_patch(
                Rectangle(
                    (lx - 0.011, ly - 0.035), 0.022, 0.07,
                    transform=axl.transAxes, fill=False,
                    edgecolor=INK, linewidth=1.4,
                    linestyle="--"
                )
            )

        elif key == "cell":
            axl.add_patch(
                Rectangle(
                    (lx - 0.011, ly - 0.035), 0.022, 0.07,
                    transform=axl.transAxes,
                    facecolor=cmap(norm(0.05)),
                    edgecolor="none"
                )
            )

        axl.text(
            lx + 0.025, ly, text,
            fontsize=11.5, color=INK, va="center"
        )

        ly -= 0.15

    # ========================================================
    # 12-6. 저장
    # ========================================================

    out = (
        OUTPUT_DIR
        / f"{kind}_vulnerability_map.png"
    )

    fig.savefig(
        out,
        dpi=300,
        facecolor="white",
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

    print("저장 완료:", out)


# ============================================================
# 13. 폭염·폭우 지도 생성
# ============================================================

print("\n폭염 지도 생성 중...")
make("heat")

print("\n폭우 지도 생성 중...")
make("rain")


# ============================================================
# 14. 격자별 점수 저장
# ============================================================

cols = [
    "region",
    "X_COORD",
    "Y_COORD",
    "type",
    "elder_share",
    "n_stores",
    "has_golf",
    "has_lego",
    "heat_score",
    "rain_score"
]

out = grid[cols].copy()

out["heat_score_pct"] = (
    out["heat_score"] * 100
)

out["rain_score_pct"] = (
    out["rain_score"] * 100
)

out.to_csv(
    OUTPUT_DIR / "grid_vulnerability_scores.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "\n저장 완료:",
    OUTPUT_DIR / "grid_vulnerability_scores.csv"
)


# ============================================================
# 15. 상권 유형별 평균
# ============================================================

print("\n[춘천 상권 유형별 평균 %]")

print(
    grid[chun]
    .groupby("type")[
        ["heat_score", "rain_score"]
    ]
    .mean()
    .mul(100)
    .round(2)
)

print(
    "\n[골프장·레고랜드 격자 수]:",
    int((special & chun).sum()),
    "(정상: 13)"
)

print("\n모든 작업이 완료되었습니다.")